# Exp 1 (reprocessed) — Experimental Data Pipeline

**Source:** `../raw_data.xlsx` — fresh raw E-Prime export dropped into `data_processing3/` to reprocess
Experiment 1 through the same Python pipeline used for Exp 2 and Exp 3 (instead of the original
manual Excel/SPSS workflow in `data_processing1/` and `data_processing2/`).

Known issues with this raw export, handled explicitly below:
1. ss1, ss17, ss45 were excluded before this export for technical problems during collection —
   already absent from the file, Step 01b filters explicitly anyway for self-documentation.
2. ss12 physically reversed their response keys for the color-identification task, so E-Prime scored
   almost every trial as wrong (0.6% accuracy). Step 01b fixes this by inverting their accuracy —
   same step as the technical-exclusion filter, since both are pre-Step-02 data-quality corrections.
3. RT trimming uses **Python ±2SD** (same as Exp 2/3) — SPSS's `2w.SPS` script has a confirmed bug
   (see `SPSS_2w_script_bug_summary.txt` in the project root): it silently keeps RT=0 trials that
   its own filter logic flags for exclusion, because `FiltRT = filt*rt` hits SPSS's
   `SYSMIS * 0 = 0` rule. Python trimming is canonical for all three experiments now.

Each checkpoint is saved as a numbered xlsx file so you can open any step in Excel and go back if needed.

| Step | File | What happened |
|------|------|---------------|
| 00 | `00_raw.xlsx` | Experimental (`Procedure[SubTrial]='pro'`) rows split from `raw_data.xlsx` |
| 01 | `01_filtered.xlsx` | One row per trial, selected columns, acc + rt added |
| 01b | `01b_fixed.xlsx` | ss1/ss17/ss45 removed (technical problems) + ss12 accuracy inverted (button-reversal fix) |
| 02 | `02_excl_acc.xlsx` | Excluded subjects below 85% accuracy threshold |
| 03 | `03_excl_likert.xlsx` | Excluded careless Likert responders (aligned with questions pipeline) |
| 04 | `04_excl_errors.xlsx` | Error trials (acc = 0) removed |
| 05 | `05_gaze_congruency.xlsx` | gaze_congruency, action_consistency, initiation_time columns added |
| 06 | `06_trimmed.xlsx` | RT outlier trimming — **Python ±2SD**, per subject × condition |
| 07 | `07_no_outliers.xlsx` | Participant-level outlier removed (mean RT > grand mean + 2 SD) |
| 08 | `08_ranova_input.xlsx` | Mean RT per subject × condition — long + wide, ready for rANOVA |
| 09 | `09_initiation_time.xlsx` | Mean initiation time per subject, action trials only (ANCOVA covariate) |
| 10 | `10_questions_data_added.xlsx` | Wide RT data + question ratings + initiation-time covariate merged |
| 11 | `11_merged_initTime_for_ANCOVA.xlsx` | Stable xlsx copy of the merged data (same content as Step 10) — kept in `exp_data/`, won't change if the pipeline re-runs |
| — | `jasp/analysis.csv` | Same data as CSV — open in JASP to analyse (canonical, overwritten every run) |
| 12 | `12_anova_effect_size.xlsx` | RT repeated-measures ANOVA + Cohen's dz effect sizes per action condition (Exp1 only, for now) |

In [1]:
import os
import pandas as pd
import numpy as np

base = r'C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data'
raw_master = r'C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\raw_data.xlsx'

## Step 00 — Split master raw export into experimental rows

`raw_data.xlsx` contains both the color-discrimination trials (`Procedure[SubTrial]='pro'`) and the
post-trial SoA question rows (`Procedure[SubTrial]='question'`). This pipeline only needs the former —
the questions pipeline does the equivalent split for the latter.

In [2]:
df = pd.read_excel(raw_master)
print(f'Master raw: {df.shape[0]} rows × {df.shape[1]} cols')

exp_rows = df[df['Procedure[SubTrial]'] == 'pro'].copy()
print(f'pro (experimental) rows: {len(exp_rows)}')

out = f'{base}\\00_raw.xlsx'
exp_rows.to_excel(out, index=False)
print(f'Saved: {out}')

Master raw: 93392 rows × 78 cols
pro (experimental) rows: 85904
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\00_raw.xlsx


## Step 01 — Filter to one row per trial, select columns, add acc/rt

**Filters applied** (identical logic to Exp 2/Exp 3):
1. `Procedure[SubTrial]` = `'pro'`
2. `Running[Trial]` = `'Exp'` — excludes practice trials and question-replay rows
3. `pic` in `['6L','6R']` — keeps only the last gaze-shift frame, where the response was captured

**acc/rt logic:** `Running[SubTrial]` says which side was last shown (ListLeft or ListRight); the
matching `ColorSlideLeft`/`ColorSlideRight` columns hold the actual response data. Using `combine_first`
here would be wrong — the inactive side's ACC defaults to 0 (not null), so the side must be selected
explicitly via `Running[SubTrial]`, exactly as Exp 2/3 do it with `np.where`.

Column selection matches Exp 2/3's `exp_data` keep-list exactly (`Handedness`, not `Age`/`Sex` —
those come from the questions pipeline's demographics at the Step 10 merge, same as Exp 2/3).

In [3]:
df = pd.read_excel(f'{base}\\00_raw.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

df = df[df['Procedure[SubTrial]'] == 'pro'].copy()
print(f'After Procedure[SubTrial]=pro: {len(df)} rows')
df = df[df['Running[Trial]'] == 'Exp'].copy()
print(f'After Running[Trial]=Exp:      {len(df)} rows')
df = df[df['pic'].isin(['6L', '6R'])].copy()
print(f'After pic in [6L, 6R]:         {len(df)} rows')

n_subjects_raw = df['Subject'].nunique()
print(f'Subjects: {n_subjects_raw}  |  Expected: {n_subjects_raw} x 168 = {n_subjects_raw*168}')

trial_counts = df.groupby('Subject').size()
bad = trial_counts[trial_counts != 168]
print('Subjects with != 168 trials:', 'none' if bad.empty else bad.to_dict())

keep = [
    'Subject', 'Handedness', 'blue', 'green',
    'colorLeft', 'colorRight',
    'ColorSlideLeft.ACC', 'ColorSlideLeft.RESP', 'ColorSlideLeft.RT',
    'ColorSlideRight.ACC', 'ColorSlideRight.RESP', 'ColorSlideRight.RT',
    'corR', 'Exp.Sample', 'GazeColorConsistency',
    'Procedure[Trial]', 'SlideXeroA.RT', 'XeroDur',
    'pic', 'Running[SubTrial]'
]
df = df[keep].copy()
print(f'Columns selected: {df.shape[1]}')

left = df['Running[SubTrial]'] == 'ListLeft'
right = df['Running[SubTrial]'] == 'ListRight'
df['acc'] = np.where(left, df['ColorSlideLeft.ACC'], np.where(right, df['ColorSlideRight.ACC'], np.nan))
df['rt'] = np.where(left, df['ColorSlideLeft.RT'], np.where(right, df['ColorSlideRight.RT'], np.nan))
print(f'acc nulls: {df["acc"].isna().sum()}  |  rt nulls: {df["rt"].isna().sum()}')

acc_per_sub = df.groupby('Subject')['acc'].mean().mul(100).round(2).reset_index()
acc_per_sub.columns = ['Subject', 'ACC_%']
acc_per_sub['flag'] = acc_per_sub['ACC_%'].apply(lambda x: '*** <80%' if x < 80 else ('* <85%' if x < 85 else ''))
print(acc_per_sub.to_string(index=False))

out = f'{base}\\01_filtered.xlsx'
df.to_excel(out, index=False)
print(f'Saved: {out}')

Loaded: 85904 rows × 78 cols
After Procedure[SubTrial]=pro: 85904 rows
After Running[Trial]=Exp:      61152 rows
After pic in [6L, 6R]:         8736 rows
Subjects: 52  |  Expected: 52 x 168 = 8736
Subjects with != 168 trials: none
Columns selected: 20
acc nulls: 0  |  rt nulls: 0
 Subject  ACC_%     flag
       2  89.29         
       3  64.88 *** <80%
       4  90.48         
       5  98.21         
       6  47.62 *** <80%
       7  94.64         
       8  61.90 *** <80%
       9  89.88         
      10   0.00 *** <80%
      11  88.10         
      12   0.60 *** <80%
      13  94.64         
      14  99.40         
      15  48.21 *** <80%
      16  97.02         
      18  93.45         
      19  77.38 *** <80%
      20  93.45         
      21  67.26 *** <80%
      22  98.21         
      23  86.90         
      24  89.29         
      25  88.10         
      26  97.02         
      27  90.48         
      28  98.21         
      29  97.62         
      30  89.29    

## Step 01b — Data quality fixes: technical exclusions + ss12 button-reversal

Two corrections applied together here, both needed before accuracy is used for anything:

**ss1, ss17, ss45** — excluded before this raw export for technical problems during collection
(screen going black mid-session, computer restart, session aborted — see `SS_list.xlsx`). Already
absent from `raw_data.xlsx`; filtered explicitly anyway so the exclusion is documented in the
pipeline itself rather than relying on them silently having no rows.

**ss12** — physically reversed which key they used for which color for virtually the entire session.
Evidence: `RESP == corR` in only 1/168 trials (a near-total, consistent reversal — not random
error), and the experimenter's live session note in `SS_list.xlsx` recorded ~99% accuracy for
ss12, matching the post-fix 99.4% almost exactly. Fix: invert `acc` for ss12's rows only
(`1 - acc`); RT is untouched. This must happen **before** Step 02's accuracy threshold, or ss12
gets wrongly excluded on their pre-fix ~0.6% accuracy.

In [5]:
df = pd.read_excel(f'{base}\\01_filtered.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

TECHNICAL_EXCLUDED = [1, 17, 45]
present = [s for s in TECHNICAL_EXCLUDED if s in df['Subject'].unique()]
print(f'Technical-exclusion subjects found: {present if present else "none (already absent)"}')
n_before = len(df)
df = df[~df['Subject'].isin(TECHNICAL_EXCLUDED)].copy()
print(f'Rows removed (technical exclusion): {n_before - len(df)}')

resp12 = df.loc[df['Subject'] == 12]
match_check = (resp12['ColorSlideLeft.RESP'].combine_first(resp12['ColorSlideRight.RESP']) == resp12['corR'])
print(f'ss12: RESP==corR in {match_check.sum()}/{len(resp12)} trials before fix (near-total reversal)')
before = df.loc[df['Subject'] == 12, 'acc'].mean() * 100
df.loc[df['Subject'] == 12, 'acc'] = 1 - df.loc[df['Subject'] == 12, 'acc']
after = df.loc[df['Subject'] == 12, 'acc'].mean() * 100
print(f'ss12 accuracy before fix: {before:.1f}%  |  after fix: {after:.1f}%  (experimenter note: ~99%)')

out = f'{base}\\01b_fixed.xlsx'
df.to_excel(out, index=False)
print(f'Saved: {out}')

Loaded: 8736 rows × 22 cols
Technical-exclusion subjects found: none (already absent)
Rows removed (technical exclusion): 0
ss12: RESP==corR in 1/168 trials before fix (near-total reversal)
ss12 accuracy before fix: 0.6%  |  after fix: 99.4%  (experimenter note: ~99%)
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\01b_fixed.xlsx


## Step 02 — Exclude participants below 85% accuracy

Threshold: **85%** — same as Exp 2/3 (confirmed with Armina: all three experiments use 85%).
ss40 (84.52%) is the only subject this threshold affects on this dataset — excluded here, would be
kept at 80%.

This step no longer writes an exclusion list for the questions pipeline to read — questions
doesn't need it. Low-accuracy subjects naturally drop out later when Step 10's merge keeps only
subjects present in this (exp_data's) final sample.

In [6]:
df = pd.read_excel(f'{base}\\01b_fixed.xlsx')
acc_per_sub = df.groupby('Subject')['acc'].mean().mul(100).round(2)
THRESHOLD = 85.0
exclude = acc_per_sub[acc_per_sub < THRESHOLD]
keep_subs = acc_per_sub[acc_per_sub >= THRESHOLD].index
print(f'Threshold: {THRESHOLD}%')
print(f'Excluded subjects ({len(exclude)}):')
for sub, acc in exclude.sort_values().items():
    print(f'  ss {sub}: {acc:.2f}%')
df2 = df[df['Subject'].isin(keep_subs)].copy()
print(f'Remaining: {df2["Subject"].nunique()} subjects, {len(df2)} trials')
out = f'{base}\\02_excl_acc.xlsx'
df2.to_excel(out, index=False)
print(f'Saved: {out}')
excl_list = sorted(exclude.index.tolist())
with open(f'{base}\\02_acc_excluded_subjects.txt', 'w') as f:
    f.write(','.join(str(s) for s in excl_list))
print(f'ACC-excluded subject list saved: {excl_list}')

Threshold: 85.0%
Excluded subjects (8):
  ss 10: 0.00%
  ss 6: 47.62%
  ss 15: 48.21%
  ss 8: 61.90%
  ss 3: 64.88%
  ss 21: 67.26%
  ss 19: 77.38%
  ss 40: 84.52%
Remaining: 44 subjects, 7392 trials
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\02_excl_acc.xlsx
ACC-excluded subject list saved: [3, 6, 8, 10, 15, 19, 21, 40]


## Step 03 — Exclude careless Likert responders

Reads the excluded-subject list `questions/pipeline.ipynb` Step 03 writes directly
(`03_likert_excluded_subjects.txt`) — no recomputation from the report, no hardcoded list.

**Excluded (2):** ss 18, ss 23 — >75% identical Likert responses, near-zero range across conditions
**Kept despite flag:** ss 55 (range = 1.88) — discriminates between conditions

In [8]:
with open(f'{os.path.dirname(base)}\\questions\\03_likert_excluded_subjects.txt') as f:
    content = f.read().strip()
    LIKERT_EXCLUDED = [int(x) for x in content.split(',') if x] if content else []
print(f'Read from questions pipeline: {LIKERT_EXCLUDED}')

df = pd.read_excel(f'{base}\\02_excl_acc.xlsx')
df3 = df[~df['Subject'].isin(LIKERT_EXCLUDED)].copy()
print(f'Remaining subjects: {df3["Subject"].nunique()}  |  Remaining trials: {len(df3)}')
out = f'{base}\\03_excl_likert.xlsx'
df3.to_excel(out, index=False)
print(f'Saved: {out}')

Read from questions pipeline: [18, 23]
Remaining subjects: 42  |  Remaining trials: 7056
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\03_excl_likert.xlsx


## Step 04 — Remove error trials

Error trials: `acc = 0` — participant pressed the wrong key or timed out (post ss12-fix).

In [9]:
df = pd.read_excel(f'{base}\\03_excl_likert.xlsx')
n_before = len(df)
n_errors = (df['acc'] == 0).sum()
print(f'Error trials (acc = 0): {n_errors}  ({n_errors/n_before*100:.2f}%)')
df4 = df[df['acc'] == 1].copy()
print(f'Remaining: {df4["Subject"].nunique()} subjects, {len(df4)} trials')
out = f'{base}\\04_excl_errors.xlsx'
df4.to_excel(out, index=False)
print(f'Saved: {out}')

Error trials (acc = 0): 413  (5.85%)
Remaining: 42 subjects, 6643 trials
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\04_excl_errors.xlsx


## Step 05 — Calculate gaze_congruency, action_consistency, initiation_time

Same direct-map logic as Exp 2/3 (`action_consistency` = relabel of `Procedure[Trial]`, not a
computed actual-vs-wanted comparison — verified identical on this data, see earlier discussion).

| Column | Logic |
|--------|-------|
| `actual_color_location` | `colorLeft == "white"` → "right", else "left" |
| `actual_gaze_dir` | `Running[SubTrial] == "ListLeft"` → "left", else "right" |
| `gaze_congruency` | `actual_color_location == actual_gaze_dir` → "congruent", else "incongruent" |
| `action_consistency` | `actionC` → "consistent", `actionNC` → "inconsistent", `controlLeft`/`controlRight` → "control" |
| `initiation_time` | `SlideXeroA.RT` (action trials) filled with `XeroDur` (control trials) |

In [10]:
df = pd.read_excel(f'{base}\\04_excl_errors.xlsx')
df['actual_color_location'] = np.where(df['colorLeft'] == 'white', 'right', 'left')
df['actual_gaze_dir'] = np.where(df['Running[SubTrial]'] == 'ListLeft', 'left', 'right')
df['gaze_congruency'] = np.where(df['actual_color_location'] == df['actual_gaze_dir'], 'congruent', 'incongruent')
df['action_consistency'] = df['Procedure[Trial]'].map({
    'actionC': 'consistent', 'actionNC': 'inconsistent',
    'controlLeft': 'control', 'controlRight': 'control',
})
df['initiation_time'] = df['SlideXeroA.RT'].fillna(df['XeroDur'])
print('gaze_congruency:', df['gaze_congruency'].value_counts().to_dict())
print('action_consistency:', df['action_consistency'].value_counts().to_dict())
print('initiation_time missing:', df['initiation_time'].isna().sum())
out = f'{base}\\05_gaze_congruency.xlsx'
df.to_excel(out, index=False)
print(f'Saved: {out}  shape={df.shape}')

gaze_congruency: {'incongruent': 3343, 'congruent': 3300}
action_consistency: {'consistent': 2230, 'inconsistent': 2221, 'control': 2192}
initiation_time missing: 0
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\05_gaze_congruency.xlsx  shape=(6643, 27)


## Step 06 — RT outlier trimming (Python ±2SD)

For each `Subject × gaze_congruency × action_consistency` cell, drop trials more than 2 SD from
that cell's own mean, then take the mean of what remains. No SPSS round-trip, no numeric w1/w2
recoding — grouping is done directly on the string labels.

In [11]:
df = pd.read_excel(f'{base}\\05_gaze_congruency.xlsx')
grp = df.groupby(['Subject', 'gaze_congruency', 'action_consistency'])['rt']
stats = grp.agg(['mean', 'std']).reset_index()
df = df.merge(stats, on=['Subject', 'gaze_congruency', 'action_consistency'])
keep_mask = (df['rt'] >= df['mean'] - 2*df['std']) & (df['rt'] <= df['mean'] + 2*df['std'])
n_before = len(df)
trimmed = df[keep_mask].copy()
n_after = len(trimmed)
print(f'Trials before trim: {n_before}')
print(f'Trials after +/-2SD trim: {n_after}  (removed {n_before-n_after}, {(n_before-n_after)/n_before*100:.2f}%)')
cellmeans = trimmed.groupby(['Subject', 'gaze_congruency', 'action_consistency'])['rt'].mean().reset_index()
cellmeans.columns = ['sub', 'gaze_congruency', 'action_consistency', 'rt_mean']
print(f'Cell means shape: {cellmeans.shape}')
out = f'{base}\\06_trimmed.xlsx'
cellmeans.to_excel(out, index=False)
print(f'Saved: {out}')

Trials before trim: 6643
Trials after +/-2SD trim: 6325  (removed 318, 4.79%)
Cell means shape: (252, 4)
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\06_trimmed.xlsx


## Step 07 — Participant-level mean RT outlier removal

Excludes participants whose overall mean (mean of their 6 cell means) exceeds grand mean + 2 SD.

**ss12 is excluded here** — mean RT 705.6ms vs. a 702.1ms cutoff, a 3.5ms margin. This is unrelated
to the button-reversal fix (that was correctly applied — ss12 passed every earlier step on their
true ~99% accuracy) and is a genuine, if marginal, RT-based exclusion. Per prior discussion: if it
drops at this final step, it drops — this is accepted as the final result.

In [12]:
df = pd.read_excel(f'{base}\\06_trimmed.xlsx')
subj_means = df.groupby('sub')['rt_mean'].mean().reset_index()
subj_means.columns = ['sub', 'overall_mean_rt']
group_mean = subj_means['overall_mean_rt'].mean()
group_sd = subj_means['overall_mean_rt'].std()
cutoff = group_mean + 2*group_sd
excluded = subj_means[subj_means['overall_mean_rt'] > cutoff]
exclude_ids = excluded['sub'].tolist()
print(f'Grand mean RT : {group_mean:.1f} ms')
print(f'Grand SD      : {group_sd:.1f} ms')
print(f'Cutoff        : {cutoff:.1f} ms (mean + 2 SD)')
print(f'Excluded ({len(excluded)} subject(s)):')
for _, row in excluded.iterrows():
    print(f'  ss {int(row["sub"])}: mean RT = {row["overall_mean_rt"]:.1f} ms (+{row["overall_mean_rt"]-cutoff:.1f} ms over cutoff)')
clean = df[~df['sub'].isin(exclude_ids)].copy()
print(f'Remaining: {clean["sub"].nunique()} subjects, {len(clean)} rows')
out = f'{base}\\07_no_outliers.xlsx'
clean.to_excel(out, index=False)
print(f'Saved: {out}')

Grand mean RT : 522.7 ms
Grand SD      : 89.7 ms
Cutoff        : 702.1 ms (mean + 2 SD)
Excluded (2 subject(s)):
  ss 2: mean RT = 747.4 ms (+45.4 ms over cutoff)
  ss 12: mean RT = 705.6 ms (+3.5 ms over cutoff)
Remaining: 40 subjects, 240 rows
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\07_no_outliers.xlsx


## Step 08 — Reshape to long + wide

In [13]:
df = pd.read_excel(f'{base}\\07_no_outliers.xlsx')
long = df[['sub', 'gaze_congruency', 'action_consistency', 'rt_mean']].copy()
n_subs = long['sub'].nunique()
print(f'Long format: {len(long)} rows ({n_subs} subjects x 6 conditions = {n_subs*6})')
col_map = {
    ('incongruent', 'inconsistent'): 'incong_incons', ('incongruent', 'consistent'): 'incong_cons', ('incongruent', 'control'): 'incong_ctrl',
    ('congruent', 'inconsistent'): 'cong_incons', ('congruent', 'consistent'): 'cong_cons', ('congruent', 'control'): 'cong_ctrl',
}
wide = df.pivot(index='sub', columns=['gaze_congruency', 'action_consistency'], values='rt_mean')
wide.columns = [col_map[c] for c in wide.columns]
wide = wide.reset_index()
print(f'Wide format: {wide.shape[0]} subjects x {wide.shape[1]} columns')
out = f'{base}\\08_ranova_input.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as writer:
    long.to_excel(writer, sheet_name='long', index=False)
    wide.to_excel(writer, sheet_name='wide', index=False)
print(f'Saved: {out}  (sheets: long, wide)')

Long format: 240 rows (40 subjects x 6 conditions = 240)
Wide format: 40 subjects x 7 columns
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\08_ranova_input.xlsx  (sheets: long, wide)


## Step 09 — Mean initiation time per subject (ANCOVA covariate)

`initiation_time` is `SlideXeroA.RT` on action trials (a real behavioral measure) but `XeroDur` on
control trials (an experimenter-imposed random delay). Averaging across both would mix a genuine
measure with arbitrary noise, and there's no equivalent "prediction" construct in control trials for
the covariate to represent — so this computes the mean **only across action trials** (consistent +
inconsistent). Any ANCOVA using this covariate should itself be restricted to the action-condition
cells (`cong_cons`, `cong_incons`, `incong_cons`, `incong_incons`) — not the `*_ctrl` columns.

In [14]:
df = pd.read_excel(f'{base}\\05_gaze_congruency.xlsx')
action_only = df[df['action_consistency'].isin(['consistent', 'inconsistent'])]
init_time = action_only.groupby('Subject')['initiation_time'].mean().reset_index()
init_time.columns = ['sub', 'mean_initiation_time']
print(f'Computed for {len(init_time)} subjects (from {len(action_only)} action trials)')
print(init_time.describe().to_string())
out = f'{base}\\09_initiation_time.xlsx'
init_time.to_excel(out, index=False)
print(f'Saved: {out}')

Computed for 42 subjects (from 4451 action trials)
             sub  mean_initiation_time
count  42.000000             42.000000
mean   31.285714            523.241630
std    15.512836            260.963119
min     2.000000            249.158879
25%    20.500000            332.463303
50%    32.500000            431.397426
75%    43.750000            580.274752
max    55.000000           1365.616162
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\09_initiation_time.xlsx


## Step 10 — Merge question ratings + initiation time into wide RT data

Left join on `sub` — keeps only the 40 subjects present in the final exp data.

In [15]:
rt_wide = pd.read_excel(f'{base}\\08_ranova_input.xlsx', sheet_name='wide')
q_wide = pd.read_excel(f'{os.path.dirname(base)}\\questions\\04_aggregated.xlsx', sheet_name='wide')
init_time = pd.read_excel(f'{base}\\09_initiation_time.xlsx')
merged = rt_wide.merge(q_wide, on='sub', how='left').merge(init_time, on='sub', how='left')
print(f'Subjects in exp data:       {rt_wide["sub"].nunique()}')
print(f'Subjects in questions data: {q_wide["sub"].nunique()}')
print(f'Subjects in merged output:  {merged["sub"].nunique()}')
dropped = set(q_wide['sub']) - set(rt_wide['sub'])
if dropped:
    print(f'Dropped from questions (not in exp data): ss {sorted(dropped)}')
missing_init = merged[merged['mean_initiation_time'].isna()]
if len(missing_init):
    print(f'WARNING missing initiation time for: {missing_init["sub"].tolist()}')
print(f'Columns ({merged.shape[1]}): {list(merged.columns)}')
out = f'{base}\\10_questions_data_added.xlsx'
merged.to_excel(out, index=False)
print(f'Saved: {out}')

Subjects in exp data:       40
Subjects in questions data: 50
Subjects in merged output:  40
Dropped from questions (not in exp data): ss [2, 3, 6, 8, 10, 12, 15, 19, 21, 40]
Columns (20): ['sub', 'cong_cons', 'cong_ctrl', 'cong_incons', 'incong_cons', 'incong_ctrl', 'incong_incons', 'consistent_caused', 'consistent_control', 'consistent_predict', 'control_caused', 'control_control', 'control_predict', 'inconsistent_caused', 'inconsistent_control', 'inconsistent_predict', 'Age', 'Sex', 'Handedness', 'mean_initiation_time']
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\10_questions_data_added.xlsx


## Step 11 — Export to JASP

After opening in JASP: set `sub`, `Sex`, `Handedness` to Nominal. For the RT rANOVA, use the 6
`*_cons`/`*_incons`/`*_ctrl` columns. For the ANCOVA, use only the 4 action-condition columns with
`mean_initiation_time` as covariate — do not include the `*_ctrl` columns in that analysis.

Also saves `11_merged_initTime_for_ANCOVA.xlsx` in `exp_data/` — a stable xlsx copy of the same merged data (same content as `10_questions_data_added.xlsx`), following the numbered-checkpoint naming convention, meant for continued JASP or Python analysis work without changing if the pipeline is re-run.

In [ ]:
jasp_dir = os.path.join(base, 'jasp')
os.makedirs(jasp_dir, exist_ok=True)
csv_path = os.path.join(jasp_dir, 'analysis.csv')
merged.to_csv(csv_path, index=False, encoding='utf-8')
print(f'Saved: {csv_path}')

working_path = f'{base}\\11_merged_initTime_for_ANCOVA.xlsx'
merged.to_excel(working_path, index=False)
print(f'Saved: {working_path}  (stable xlsx copy for continued JASP/Python analysis work)')
print(f'Subjects: {merged.shape[0]}  |  Columns: {merged.shape[1]}')

## Step 12 — RT repeated-measures ANOVA + effect sizes

The standard within-experiment analysis: a 2 (`gaze_congruency`) x 3 (`action_consistency`)
repeated-measures ANOVA on `rt_mean`, plus the gaze-cueing effect size for each action condition.

- **ANOVA**: `pingouin.rm_anova`, within-subject factors `gaze_congruency` and `action_consistency`,
  subject = `sub`. Reports F, p, and partial eta-squared (`np2`) for both main effects and the
  interaction.
- **Effect size**: the gaze-cueing effect itself is `incongruent_rt - congruent_rt`, computed per
  subject, separately for each action condition (consistent / inconsistent / control). Cohen's
  **dz** (paired-samples effect size = mean difference / SD of the differences) summarizes the size
  of that effect within each condition; `pingouin.ttest(..., paired=True)` reports the matching
  paired t-test and `dz` together.

This is the "standard" analysis only — the cross-experiment comparison Armina described (comparing
this same effect, in ms and in Cohen's d, across Exp1/2/3 via a between-experiments ANOVA) is a
separate later step, not done here.

Saved: `12_anova_effect_size.xlsx` (sheets: `rm_anova`, `effect_sizes`)

In [17]:
import pingouin as pg

long = pd.read_excel(f'{base}\\08_ranova_input.xlsx', sheet_name='long')
print(f'Loaded: {long.shape[0]} rows ({long["sub"].nunique()} subjects x 6 conditions)')

aov = pg.rm_anova(
    data=long, dv='rt_mean', within=['gaze_congruency', 'action_consistency'],
    subject='sub', detailed=True, effsize='np2'
)
print(aov.round(4).to_string(index=False))


Loaded: 240 rows (40 subjects x 6 conditions)
                              Source         SS  ddof1  ddof2        MS       F  p_unc  p_GG_corr    np2    eps
                     gaze_congruency  9275.8205      1     39 9275.8205 17.0209 0.0002     0.0002 0.3038 1.0000
                  action_consistency 17987.7387      2     78 8993.8693  5.3165 0.0068     0.0143 0.1200 0.7267
gaze_congruency * action_consistency   980.1346      2     78  490.0673  0.8342 0.4381     0.4376 0.0209 0.9951


In [18]:
wide = pd.read_excel(f'{base}\\08_ranova_input.xlsx', sheet_name='wide')

conditions = ['cons', 'incons', 'ctrl']
rows = []
for cond in conditions:
    cong_col = f'cong_{cond}'
    incong_col = f'incong_{cond}'
    diff = wide[incong_col] - wide[cong_col]
    ttest = pg.ttest(wide[incong_col], wide[cong_col], paired=True)
    rows.append({
        'action_consistency': cond,
        'n': wide['sub'].nunique(),
        'mean_effect_ms': diff.mean(),
        'sd_effect_ms': diff.std(),
        't': ttest['T'].iloc[0],
        'p': ttest['p_val'].iloc[0],
        'cohen_dz': ttest['cohen_d'].iloc[0],
    })

effect_sizes = pd.DataFrame(rows)
print(effect_sizes.round(4).to_string(index=False))

out = f'{base}\\12_anova_effect_size.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as writer:
    aov.to_excel(writer, sheet_name='rm_anova', index=False)
    effect_sizes.to_excel(writer, sheet_name='effect_sizes', index=False)
print(f'Saved: {out}')


action_consistency  n  mean_effect_ms  sd_effect_ms      t      p  cohen_dz
              cons 40         15.7792       28.1530 3.5448 0.0010    0.1966
            incons 40          6.7473       33.2831 1.2821 0.2074    0.0836
              ctrl 40         14.7745       39.2367 2.3815 0.0222    0.1634
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment1\data_processing3\exp_data\12_anova_effect_size.xlsx
